In [127]:
import os
import sys
import glob
import numpy as np
from tqdm import trange
from astropy.io import fits
from astropy.table import Table, vstack
from astropy.convolution import convolve, Gaussian1DKernel
import astropy.units as u
import astropy.coordinates as coord
import matplotlib
import matplotlib.pyplot as plt
from astropy.table import Column
from tqdm import trange
import pandas as pd
import matplotlib.ticker as mticker
import fitsio
from astropy.table import Table, vstack
from astropy import units as u
from astropy.coordinates import SkyCoord
from easyquery import Query, QueryMaker
from scipy.stats import binomtest
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.colors import LogNorm
from matplotlib.colors import ListedColormap, BoundaryNorm
import h5py
from astropy.cosmology import Planck18
import glob
from matplotlib.lines import Line2D

rootdir = '/global/u1/v/virajvm/'
sys.path.append(os.path.join(rootdir, 'DESI2_LOWZ/desi_dwarfs/code'))

from desi_lowz_funcs import make_subplots, match_c_to_catalog, print_radecs, process_img
from desi_lowz_funcs import calc_normalized_dist, sdss_rgb, get_scrollable_pdfs
from desi_lowz_funcs import find_objects_nearby, print_radecs
# from construct_dwarf_galaxy_catalogs import process_sga_matches

import warnings
from astropy.wcs import FITSFixedWarning

from desi_lowz_funcs import plot_2d_dist, make_subplots

%load_ext autoreload
%autoreload 2




The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [140]:
## also load my main catalog so I can get the ra, dec values!

In [141]:
# Path to the catalog
filename = "/pscratch/sd/v/virajvm/desi_dwarf_catalogs/dr1/v1.0/desi_dr1_dwarf_catalog.fits"

# Option 1: load the MAIN extension directly as an Astropy Table
main = Table.read(filename, hdu="MAIN")


In [128]:
cont_path = "/pscratch/sd/v/virajvm/desi_dwarf_catalogs/contaminants/contaminant_flags_dr1only.fits"


In [129]:
#!/usr/bin/env python
"""
Explore example objects from each contaminant class in the DESI DR1 dwarf
contaminant-flags catalog.

For every CONTAM_CLASS present in the file, this:
  1. prints the class distribution,
  2. pulls N example rows per class (strongest evidence first),
  3. decodes each object's CONTAM_BITMASK into human-readable evidence, and
  4. prints a Legacy Survey desi-spectrum viewer URL so you can load each one.

Reference: contaminant_flags README sections 4 and 6.
"""

import numpy as np
from astropy.table import Table

# ----------------------------------------------------------------------------
# Config
# ----------------------------------------------------------------------------
CONT_PATH = "/pscratch/sd/v/virajvm/desi_dwarf_catalogs/contaminants/contaminant_flags_dr1only.fits"

N_PER_CLASS = 3        # examples to show per class
SELECT = "strongest"   # "strongest" (conclusive first) or "random"
RANDOM_SEED = 0        # only used when SELECT == "random"

SPEC_URL = "https://www.legacysurvey.org/viewer/desi-spectrum/dr1/targetid{tid}"

# Confidence ordering: most decisive first.
CONF_RANK = {"conclusive": 0, "mh-provisional": 1, "suggestive": 2, "none": 3}

# Bit -> (short name, plain-English evidence). From README section 6, plus
# bit 20 (deep-spectrum tested) noted in section 3. Bits 16/17/20 only ever
# appear in the *_mh.fits file; harmless to keep here.
BIT_INFO = {
    0:  ("FLAGA_HIGH",          "two DR1 coadds disagree on z (dchi2>=100)"),
    1:  ("FLAGA_MED",           "two DR1 coadds disagree on z (dchi2>=40)"),
    2:  ("FLAGA_LOW",           "coadd z disagreement, lower conf (dchi2>=25)"),
    3:  ("RR_STAR_COADD",       "another coadd of this fiber fit as a STAR"),
    4:  ("GAIA_STAR",           "Gaia parallax/PM + quiescent spectrum"),
    5:  ("GAIA_NEIGHBOR_BLEND", "bright Gaia neighbor 1.2-4'' + FRACFLUX>0.35"),
    6:  ("MASKBITS_BRIGHT",     "Legacy bright-star/cluster mask touches source"),
    7:  ("GC_UCD_HOSTBOUND",    "pos/vel match to a massive 50MGC galaxy"),
    8:  ("UCD_NSC_FREE",        "compact + quiescent, no host association"),
    9:  ("SPEC_HIGHZ",          "spectroscopy: true system at a different z"),
    10: ("SPEC_STAR",           "spectroscopy: stellar absorption + cool cont."),
    11: ("SINGLE_LINE_Z",       "redshift rests on a single emission line"),
    12: ("PHOT_ARTIFACT",       "single-epoch imaging / bad fit (NOBS<=1, RCHISQ>4)"),
    13: ("DIST_NO_FLOW",        "cz<3000 km/s without flow-corrected distance"),
    14: ("NOT_PRIMARY_CLEAN",   "duplicate, not primary after fix"),
    15: ("SPEC_WRONGZ_SUSPECT", "wrong-z hinted by spectral break only"),
    16: ("MH_DISCREPANT_CONF",  "Matterhorn full-depth z differs, confidently"),
    17: ("MH_DISCREPANT_UNCERTAIN", "Matterhorn z differs, uncertain at full depth"),
    18: ("QSO_TARGET",          "DESI targeted it as a QSO candidate"),
    19: ("WISE_AGN_W12",        "AGN-like WISE color (W1-W2 >= 0.8)"),
    20: ("DEEP_SPEC_TESTED",    "full-depth spectrum adjudicated"),
}

# Known interesting objects from README section 8 (one-liner each).
WORKED_EXAMPLES = {
    39627793348170704: "K/M giant on a z=0.335 group galaxy (founding blend case)",
    39627715581578169: "second blend; fit directly as an M star at full depth",
    39633259813930361: "passive z~0.39 galaxy misfit as z=0.0025",
    39627649181550105: "globular cluster of NGC 4697 (dv=8 km/s, 12 kpc)",
    39628093597419171: "GC/UCD of M84, first misread as a foreground star",
    39633063008797204: "real z=0.018 galaxy, photometry killed by a G=13.3 star 4.8'' away",
    39627760284473918: "wrong-z 0.033->0.109, later confirmed to four digits",
}


# ----------------------------------------------------------------------------
# Helpers
# ----------------------------------------------------------------------------
def decode_bits(value):
    """Return list of (name, description) for every bit set in value."""
    value = int(value)
    return [BIT_INFO.get(b, (f"BIT{b}", "undocumented"))
            for b in range(value.bit_length()) if value & (1 << b)]


def as_str_col(col):
    """Normalize a FITS string column (may be bytes) to a python str array."""
    arr = np.asarray(col)
    if arr.dtype.kind == "S":
        arr = np.char.decode(arr, "utf-8")
    return arr.astype(str)


def scalar(row, name):
    """Fetch a column value from a Row; None if absent or masked/empty."""
    if name not in row.colnames:
        return None
    v = row[name]
    if v is np.ma.masked or (np.ma.isMaskedArray(v) and v.mask):
        return None
    if isinstance(v, (str, bytes, np.str_, np.bytes_)) and str(v).strip() == "":
        return None
    return v


def pick_examples(sub, n, how, rng):
    """Choose n row indices from sub-table `sub`."""
    if how == "random":
        return np.sort(rng.permutation(len(sub))[:n])
    # "strongest": conclusive confidence first, then highest suspicion score.
    conf = as_str_col(sub["CONFIDENCE"])
    rank = np.array([CONF_RANK.get(c, 9) for c in conf])
    susp = (np.asarray(sub["SUSPICION_SCORE"], float)
            if "SUSPICION_SCORE" in sub.colnames else np.zeros(len(sub)))
    order = np.lexsort((-susp, rank))   # primary: rank asc, secondary: susp desc
    return order[:n]


def fmt_num(v):
    """Pretty-print a possibly-missing/NaN numeric value."""
    if v is None:
        return "n/a"
    try:
        fv = float(v)
    except (TypeError, ValueError):
        return str(v)
    return "n/a" if not np.isfinite(fv) else f"{fv:g}"


  

In [130]:
np.unique(cls_col)

array(['blend', 'catastrophic-z', 'clean', 'gc-ucd-nsc',
       'gc-ucd-nsc-candidate', 'photometry-corrupted', 'star', 'unvetted'],
      dtype='<U20')

In [131]:
t = Table.read(CONT_PATH)
print(f"Loaded {len(t):,} rows from:\n  {CONT_PATH}\n")
print("Columns:", ", ".join(t.colnames), "\n")


Loaded 469,715 rows from:
  /pscratch/sd/v/virajvm/desi_dwarf_catalogs/contaminants/contaminant_flags_dr1only.fits

Columns: TARGETID, Z, CONTAM_BITMASK, CONTAM_CLASS, CONFIDENCE, SUSPICION_SCORE, ANOMALY_PCT, KNN_CONTAM_PCT, FLAGA_TIER, FLAGA_OTHER_Z, Z_RELIABLE, DIST_RELIABLE, DWARF_PRIMARY_CLEAN, GAIA_MATCH 



In [161]:
def print_maskbit_summary(catalog,max_bitnum=18):
    maskbits = np.asarray(catalog['CONTAM_BITMASK'])  # replace `t` with your table name
    
    print(f"{'Bit':>4} {'Count':>10}")
    for bit in range(max_bitnum+1):  # bits 0 through 18
        count = np.count_nonzero(maskbits & (1 << bit))
        print(f"{bit:>4} {count:>10}")

    return 


BIT_NAMES = {
    0: 'FLAGA_HIGH', 1: 'FLAGA_MED', 2: 'FLAGA_LOW',
    3: 'RR_STAR_COADD', 4: 'GAIA_STAR', 5: 'GAIA_NEIGHBOR_BLEND',
    6: 'MASKBITS_BRIGHT', 7: 'GC_UCD_HOSTBOUND', 8: 'UCD_NSC_FREE',
    9: 'SPEC_HIGHZ', 10: 'SPEC_STAR', 11: 'SINGLE_LINE_Z',
    12: 'PHOT_ARTIFACT', 13: 'DIST_NO_FLOW', 14: 'NOT_PRIMARY_CLEAN',
}

def decode_bitmask(value, names=BIT_NAMES, verbose=True):
    value = int(value)
    on = [(bit, names.get(bit, f'BIT{bit}?'))
          for bit in range(value.bit_length()) if value >> bit & 1]
    if verbose:
        for bit, name in on:
            print(f"  bit {bit:2d}  {name}")
    return [name for _, name in on]


def bit_mask(catalog, bit, names=BIT_NAMES):
    """Boolean mask of objects with `bit` set in CONTAM_BITMASK.

    `bit` may be an int (bit number) or a str (a name from `names`)."""
    if isinstance(bit, str):
        name2bit = {v: k for k, v in names.items()}
        bit = name2bit[bit]                       # KeyError if name unknown
    maskbits = np.asarray(catalog['CONTAM_BITMASK'])
    return (maskbits & (1 << bit)) != 0

In [179]:
print_maskbit_summary(t,max_bitnum=20)

 Bit      Count
   0         15
   1         27
   2         21
   3          3
   4          5
   5       2158
   6      22358
   7         10
   8        496
   9         42
  10          2
  11       8121
  12      30578
  13        299
  14       3519
  15         12
  16          0
  17          0
  18      17112
  19       3073
  20          0


In [156]:
## we want to use bit 10!
 # 0:  ("FLAGA_HIGH",          "two DR1 coadds disagree on z (dchi2>=100)"),
    # 1:  ("FLAGA_MED",           "two DR1 coadds disagree on z (dchi2>=40)"),
## we want to use catastrophic-z ..but vi them to make sure ... 

#okay not all of these objects are bad ... look at the ca triplet and smooth it
## though we should highlight that redshift purity remains and we will present a model for that in upcoming



In [175]:
decode_bitmask(514, names=BIT_NAMES, verbose=True)


  bit  1  FLAGA_MED
  bit  9  SPEC_HIGHZ


['FLAGA_MED', 'SPEC_HIGHZ']

In [ ]:
## given a targetid, I want to gets its full breakdow of bit masks ... 

39627883991269971

In [182]:
mask_bit_0 = bit_mask(t, 0, names=BIT_NAMES)
mask_bit_1 = bit_mask(t, 1, names=BIT_NAMES)
mask_bit_2 = bit_mask(t, 2, names=BIT_NAMES)
mask_bit_3 = bit_mask(t, 3, names=BIT_NAMES)
mask_bit_9 = bit_mask(t, 9, names=BIT_NAMES)
mask_bit_10 = bit_mask(t, 10, names=BIT_NAMES)
cat_mask = (cls_col == "catastrophic-z")

#these are the things I want to VI!
tot_vi_mask = (mask_bit_0 | mask_bit_1 | mask_bit_2 | mask_bit_3 | mask_bit_9 | mask_bit_10 | cat_mask)

print(np.sum(tot_vi_mask))

87


In [184]:
##neeed to VI these 87 objects!

In [183]:

temp = main[np.isin(main["TARGETID"].data, t[mask_bit_1]["TARGETID"].data)]["RA","DEC"]

In [173]:
for i in range(len(temp)):
    print(temp["RA"][i], temp["DEC"][i])

198.63359324689262 -2.9499879672857783
153.99276199342765 0.17599653619948805
182.75071307022964 0.9375558831889793
212.65794276987805 1.0782048913750573
151.11254829321257 1.8948168504748995
335.07991772217815 29.248365734462258
31.722227016418962 29.698206411480154
334.8755929049449 31.01683949825671
251.74991384437513 35.48616521023909
172.30215022107046 51.45000731093544
213.2833261455016 52.03887631978704
177.84426955697413 53.412493521878474
177.91103835248467 53.47529286369308
178.20528501261572 55.02481438106504
184.28230058023044 55.87170512883437
182.00124344347117 -0.9197105094705762
221.30521010323582 -0.6455165553283758
213.97036864993424 -0.5071560688656087
217.46115078585703 -0.07531303627005491
179.84258980289596 1.3347546443082932
217.39012247322518 1.5936960813974024
218.75837040336344 1.6056817260914344
217.48059850439606 33.78047675769653
218.84969879943608 34.3072193648574
198.57357401565275 -2.8837557732471164
221.03146220456736 31.179280594334607
236.823272145946

In [5]:

cls_col = as_str_col(t["CONTAM_CLASS"])
classes, counts = np.unique(cls_col, return_counts=True)

print("=" * 72)
print("CLASS DISTRIBUTION")
print("=" * 72)
for c, n in sorted(zip(classes, counts), key=lambda x: -x[1]):
    print(f"  {c:<24} {n:>9,}  ({100 * n / len(t):5.2f}%)")
print()

rng = np.random.default_rng(RANDOM_SEED)

for c in classes:
    sub = t[cls_col == c]
    idx = pick_examples(sub, N_PER_CLASS, SELECT, rng)
    print("=" * 72)
    print(f"CLASS: {c}   (n={len(sub):,}, showing {len(idx)})")
    print("=" * 72)
    for i in idx:
        row = sub[int(i)]
        tid = int(row["TARGETID"])

        print(f"\n  TARGETID {tid}")
        print(f"    spectrum        : {SPEC_URL.format(tid=tid)}")
        print(f"    Z (catalog)     : {fmt_num(scalar(row, 'Z'))}")

        other_z = scalar(row, "FLAGA_OTHER_Z")
        if other_z is not None and np.isfinite(float(other_z)) and float(other_z) > 0:
            print(f"    FLAGA_OTHER_Z   : {fmt_num(other_z)}   (disagreeing coadd z)")
        tier = scalar(row, "FLAGA_TIER")
        if tier is not None:
            tier_s = (as_str_col(np.array([tier]))[0]
                      if np.asarray(tier).dtype.kind in "SU" else fmt_num(tier))
            print(f"    FLAGA_TIER      : {tier_s}")

        conf = as_str_col(np.array([row["CONFIDENCE"]]))[0]
        print(f"    confidence      : {conf}")
        print(f"    suspicion_score : {fmt_num(scalar(row, 'SUSPICION_SCORE'))}")
        print(f"    anomaly_pct     : {fmt_num(scalar(row, 'ANOMALY_PCT'))}")
        print(f"    knn_contam_pct  : {fmt_num(scalar(row, 'KNN_CONTAM_PCT'))}")
        print(f"    gaia_match      : {fmt_num(scalar(row, 'GAIA_MATCH'))}")
        for util in ("Z_RELIABLE", "DIST_RELIABLE", "DWARF_PRIMARY_CLEAN"):
            if util in sub.colnames:
                print(f"    {util:<15} : {int(row[util])}")

        bm = int(row["CONTAM_BITMASK"])
        print(f"    bitmask ({bm}) :")
        bits = decode_bits(bm)
        if bits:
            for name, desc in bits:
                print(f"        - {name}: {desc}")
        else:
            print("        (no bits set)")

        if tid in WORKED_EXAMPLES:
            print(f"    >> worked example: {WORKED_EXAMPLES[tid]}")
    print()



Loaded 469,715 rows from:
  /pscratch/sd/v/virajvm/desi_dwarf_catalogs/contaminants/contaminant_flags_dr1only.fits

Columns: TARGETID, Z, CONTAM_BITMASK, CONTAM_CLASS, CONFIDENCE, SUSPICION_SCORE, ANOMALY_PCT, KNN_CONTAM_PCT, FLAGA_TIER, FLAGA_OTHER_Z, Z_RELIABLE, DIST_RELIABLE, DWARF_PRIMARY_CLEAN, GAIA_MATCH 

CLASS DISTRIBUTION
  unvetted                   444,874  (94.71%)
  clean                       19,978  ( 4.25%)
  photometry-corrupted         4,270  ( 0.91%)
  gc-ucd-nsc-candidate           492  ( 0.10%)
  blend                           62  ( 0.01%)
  catastrophic-z                  20  ( 0.00%)
  gc-ucd-nsc                      10  ( 0.00%)
  star                             9  ( 0.00%)

CLASS: blend   (n=62, showing 3)

  TARGETID 39627715581576842
    spectrum        : https://www.legacysurvey.org/viewer/desi-spectrum/dr1/targetid39627715581576842
    Z (catalog)     : 0.00222588
    FLAGA_OTHER_Z   : 0.35151   (disagreeing coadd z)
    FLAGA_TIER      : 1
    confidence

In [ ]:
## ok these flags are pretty good!! I like them, but they are flagging some real objects, need to look into this more but very promising

## 

In [45]:
# Path to the catalog
filename = "/pscratch/sd/v/virajvm/desi_dwarf_catalogs/dr1/v1.0/desi_dr1_dwarf_catalog.fits"

# Option 1: load the MAIN extension directly as an Astropy Table
samp = Table.read(filename, hdu="MAIN")

samp_fspec = Table.read("/pscratch/sd/v/virajvm/catalog_dr1_dwarfs/desi_dr1_dwarfs.fits")


In [44]:
print(len(samp))

469715


In [46]:
len(samp_fspec)

461488

In [ ]:
## I see there is a slighlt difference between the two!

In [32]:
missing_tgids = np.loadtxt("/pscratch/sd/v/virajvm/desi_dwarf_catalogs/dr1/v1.0/missing_fastspec_targetids.txt",dtype=int)

In [33]:
missing_tgids[:5]

array([39627328841582075, 39627339843247169, 39627345564271632,
       39627351125918747, 39627374890846555])

In [34]:
len(missing_tgids)

8227

In [35]:
mask = np.isin(samp["TARGETID"].data, missing_tgids)
missing = samp[mask]

In [36]:
import fitsio, numpy as np
from astropy.table import Table


In [38]:
redux =  "/global/cfs/cdirs/desi/spectro/redux"

In [39]:
for row in missing[:50]:
    s, p, h = row['SURVEY'], row['PROGRAM'], row['HEALPIX']
    rr = f"{redux}/iron/healpix/{s}/{p}/{h//100}/{h}/redrock-{s}-{p}-{h}.fits"
    rid = fitsio.read(rr, 'REDSHIFTS', columns='TARGETID')
    print(row['TARGETID'], s, p, h, 'PRESENT' if row['TARGETID'] in rid else 'ABSENT')

##so these are not in the 

39627328841582075 main bright 36263 PRESENT
39627339843247169 main bright 17408 PRESENT
39627345564271632 main bright 17410 PRESENT
39627351125918747 main bright 16834 PRESENT
39627374890846555 main bright 36297 PRESENT
39627380771266209 main bright 36214 PRESENT
39627385896704008 main bright 17417 PRESENT
39627392397873996 main bright 22538 PRESENT
39627397573642096 main bright 36539 PRESENT
39627409594517156 main bright 36285 PRESENT
39627415030334208 main bright 36542 PRESENT
39627415504291893 main bright 36301 PRESENT
39627415621734223 main bright 22562 PRESENT
39627421288242211 main bright 36321 PRESENT
39627427177038707 main bright 36312 PRESENT
39627438694597362 main bright 36633 PRESENT
39627444151393062 main bright 17456 PRESENT
39627444537262965 main bright 36634 PRESENT
39627444788921279 main bright 36309 PRESENT
39627444889588465 main bright 22554 PRESENT
39627450006637130 main bright 17445 PRESENT
39627450392515719 main bright 36634 PRESENT
39627450631591426 main bright 36

In [92]:
temp = Table.read("/pscratch/sd/v/virajvm/legacy_galaxies/catalogs/randoms_500k.fits")

In [57]:
np.percentile(temp["PSFSIZE_G"].data, 95), np.percentile(temp["PSFSIZE_R"].data, 95), np.percentile(temp["PSFSIZE_Z"].data, 95)

(np.float32(2.1233733), np.float32(1.9206725), np.float32(1.6365495))

In [86]:
from astropy.table import hstack

In [95]:
## we need to load our dwarf galaxy catalog!!

#restrict galaxies to a total size less then 45'' and larger than 3 ''

# Path to the catalog
filename = "/pscratch/sd/v/virajvm/desi_dwarf_catalogs/dr1/v1.0/desi_dr1_dwarf_catalog.fits"

# Option 1: load the MAIN extension directly as an Astropy Table
main = Table.read(filename, hdu="MAIN")
trac = Table.read(filename, hdu="TRACTOR")



In [101]:
gal_cat = main[(main["MAG_R"] < 20) & (main["DWARF_MASKBIT"] == 0) & (main["R50_R"] > 3) & (main["R50_R"] < 45) & (main["DWARF_PRIMARY"] == True)]
trac_cat = trac[(main["MAG_R"] < 20) & (main["DWARF_MASKBIT"] == 0) & (main["R50_R"] > 3) & (main["R50_R"] < 45) & (main["DWARF_PRIMARY"] == True)]



In [102]:
len(gal_cat), len(trac_cat)

(56585, 56585)

In [103]:
gal_cat = hstack([gal_cat, trac_cat["BRICKNAME"]] ) 

In [104]:
gal_cat[:2]

TARGETID,SURVEY,PROGRAM,HEALPIX,Z,DELTACHI2,ZWARN,Z_CMB,RA,DEC,RA_TARGET,DEC_TARGET,DESINAME,LUMI_DIST_MPC,LOG_MSTAR_M24,LOG_MSTAR_M24_ERR,MAG_G,MAG_R,MAG_Z,MAG_G_TARGET,MAG_R_TARGET,MAG_Z_TARGET,SAMPLE,DWARF_MASKBIT,MSTAR_MASKBIT,MAG_TYPE,PHOTOMETRY_UPDATED,R50_R,SHAPE_PARAMS,IN_SGA_2020,ASSOCIATED_TARGETIDS,DWARF_PRIMARY_TARGETID,DWARF_PRIMARY,DIST_SOURCE,PROPERTY_SOURCE_TARGETID,BRICKNAME
int64,bytes7,bytes6,int32,float64,float64,int32,float64,float64,float64,float64,float64,bytes22,float64,float64,float64,float64,float64,float64,float64,float64,float64,bytes10,int32,int32,bytes13,bool,float64,float64[2],bool,object,int64,bool,bytes10,int64,bytes8
39627066986994125,sv1,bright,34722,0.032918604316881296,98.85089745232835,0,0.03264401958193863,58.82791003142589,-31.167992883134954,58.82791003142589,-31.167992883134954,DESI J058.8279-31.1679,148.21792602539062,8.833045959472656,0.04251172982109555,18.319110870361328,17.939651489257812,17.730533599853516,18.319110870361328,17.939651489257812,17.730533599853516,BGS_BRIGHT,0,0,TRACTOR_OG,False,7.116967678070068,0.6592394709587097 .. -87.19556427001953,False,[39627066986994125],39627066986994125,True,V_CMB,39627066986994125,0587m312
39627077380476509,sv1,bright,34724,0.058663264478969857,578.5851658955216,0,0.05841561324325051,60.776671107841956,-30.813363479786677,60.776671107841956,-30.813363479786677,DESI J060.7766-30.8133,270.1764221191406,8.923419952392578,0.03546098828559918,19.113502502441406,18.790576934814453,18.60342788696289,19.113502502441406,18.790576934814453,18.60342788696289,BGS_BRIGHT,0,0,TRACTOR_OG,False,4.0237812995910645,0.6933448910713196 .. -81.53559112548828,False,[39627077380476509],39627077380476509,True,V_CMB,39627077380476509,0608m307


In [105]:
import os
import numpy as np
import fitsio
import healpy as hp
from astropy.table import Table, vstack, unique, join

# --- your catalog: must have TARGETID, RA, DEC ---
# adjust if your columns are TARGET_RA / TARGET_DEC:
ra  = np.asarray(gal_cat["RA"],  dtype="f8")
dec = np.asarray(gal_cat["DEC"], dtype="f8")

TRACTOR_DIR = ("/global/cfs/cdirs/desi/public/dr1/vac/dr1/"
               "lsdr9-photometry/iron/v1.1/observed-targets/tractorphot")

NSIDE = 4
hpix = hp.ang2pix(NSIDE, ra, dec, lonlat=True, nest=True)   # nested!

cols  = ["TARGETID", "PSFSIZE_G", "PSFSIZE_R", "PSFSIZE_Z"]
want  = np.unique(np.asarray(gal_cat["TARGETID"]))

rows = []
for px in np.unique(hpix):
    print(px)
    fn = os.path.join(TRACTOR_DIR, f"tractorphot-nside4-hp{px:03d}-iron.fits")
    if not os.path.isfile(fn):
        # no LS/DR9 sources in that pixel for this VAC; skip
        continue
    t = Table(fitsio.read(fn, columns=cols))
    t = t[np.isin(t["TARGETID"], want)]
    rows.append(t)

phot = unique(vstack(rows), keys="TARGETID")   # one row per TARGETID in tractorphot
merged = join(gal_cat, phot, keys="TARGETID", join_type="left")  # keeps all your rows

n_missing = np.sum(merged["PSFSIZE_G"].mask) if hasattr(merged["PSFSIZE_G"], "mask") else 0
print(f"{len(merged)} objects, {n_missing} with no LS/DR9 match")

0
2
3
8
9
10
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
52
53
58
59
62
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
82
85
87
88
89
90
91
99
100
101
102
103
104
105
106
107
108
109
110
111
117
120
121
122
123
124
126
127
135
141
142
143
151
159
175
189
191
56585 objects, 2 with no LS/DR9 match


In [106]:
merged = merged[2:]

In [107]:
merged

TARGETID,SURVEY,PROGRAM,HEALPIX,Z,DELTACHI2,ZWARN,Z_CMB,RA,DEC,RA_TARGET,DEC_TARGET,DESINAME,LUMI_DIST_MPC,LOG_MSTAR_M24,LOG_MSTAR_M24_ERR,MAG_G,MAG_R,MAG_Z,MAG_G_TARGET,MAG_R_TARGET,MAG_Z_TARGET,SAMPLE,DWARF_MASKBIT,MSTAR_MASKBIT,MAG_TYPE,PHOTOMETRY_UPDATED,R50_R,SHAPE_PARAMS,IN_SGA_2020,ASSOCIATED_TARGETIDS,DWARF_PRIMARY_TARGETID,DWARF_PRIMARY,DIST_SOURCE,PROPERTY_SOURCE_TARGETID,BRICKNAME,PSFSIZE_G,PSFSIZE_R,PSFSIZE_Z
int64,bytes7,bytes6,int32,float64,float64,int32,float64,float64,float64,float64,float64,bytes22,float64,float64,float64,float64,float64,float64,float64,float64,float64,bytes10,int32,int32,bytes13,bool,float64,float64[2],bool,object,int64,bool,bytes10,int64,bytes8,float32,float32,float32
2706072053743616,main,dark,27162,0.02067871920862875,51.423833825741895,0,0.02185958059324311,147.4777724465266,-4.676317817295627,147.4777724465266,-4.676317817295627,DESI J147.4777-04.6763,98.4670639038086,8.338191032409668,0.02161700759700041,19.767858505249023,19.131961822509766,18.707191467285156,19.767858505249023,19.131961822509766,18.707191467285156,LOWZ,0,0,TRACTOR_OG,False,3.3400158882141113,0.8321884870529175 .. -77.67112731933594,False,[2706072053743616],2706072053743616,True,V_CMB,2706072053743616,1475m047,1.8973777,1.4541191,1.1051942
2706102982541312,main,dark,26141,0.017032653316364645,52.587958217802225,0,0.018156770780494602,194.32707696171147,-3.5259274788974393,194.32707696171147,-3.5259274788974393,DESI J194.3270-03.5259,81.5626220703125,8.096750259399414,0.02525354129101601,19.9845027923584,19.363792419433594,18.983266830444336,19.9845027923584,19.363792419433594,18.983266830444336,LOWZ,0,0,TRACTOR_OG,False,3.5016419887542725,0.8836742639541626 .. -33.543739318847656,False,[2706102982541312],2706102982541312,True,V_CMB,2706102982541312,1943m035,1.3323532,1.3068316,1.2971201
2706154996105216,main,dark,23102,0.026127110560297866,78.27858669866691,0,0.02564987549513864,55.44840405171893,-1.1841882389875291,55.44840405171893,-1.1841882389875291,DESI J055.4484-01.1841,115.86506652832031,8.417013168334961,0.015496369564586361,20.011428833007812,19.33967399597168,18.971805572509766,20.011428833007812,19.33967399597168,18.971805572509766,LOWZ,0,0,TRACTOR_OG,False,3.5959646701812744,0.47487756609916687 .. 63.821685791015625,False,[2706154996105216],2706154996105216,True,V_CMB,2706154996105216,0553m012,1.7105765,1.2939546,1.2036133
2706186377887744,main,dark,21865,0.03483788702310787,88.01792028345335,0,0.035780520361125,125.93542728709848,-0.0590019387959595,125.93542728709848,-0.0590019387959595,DESI J125.9354000.-590,162.83070373535156,8.681873321533203,0.018363662534729992,19.930143356323242,19.27661895751953,18.882261276245117,19.930143356323242,19.27661895751953,18.882261276245117,LOWZ,0,0,TRACTOR_OG,False,3.67437744140625,0.40696120262145996 .. -47.35565948486328,False,[2706186377887744],2706186377887744,True,V_CMB,2706186377887744,1258p000,1.3402432,1.2273034,1.3054327
2706247711195136,main,dark,27653,0.020029654061365473,40.72983623316395,0,0.0212375641784448,181.53274336777432,2.4489531897059846,181.53274336777432,2.4489531897059846,DESI J181.5327+02.4489,95.62100982666016,8.326117515563965,0.023986635610781366,19.847965240478516,19.18031120300293,18.807758331298828,19.847965240478516,19.18031120300293,18.807758331298828,LOWZ,0,0,TRACTOR_OG,False,3.303351402282715,0.8462779521942139 .. -67.152587890625,False,[2706247711195136],2706247711195136,True,V_CMB,2706247711195136,1816p025,2.330694,1.3881552,1.2428343
2706317525385216,main,dark,17899,0.016681511342151477,99.7627397031174,0,0.015739788281099898,29.443161274980465,5.4525645136574665,29.443161274980465,5.4525645136574665,DESI J029.4431+05.4525,70.57759094238281,8.093667984008789,0.017058261913048385,19.649059295654297,19.029748916625977,18.647863388061523,19.649059295654297,19.029748916625977,18.647863388061523,LOWZ,0,0,TRACTOR_OG,False,3.2249057292938232,0.7416470646858215 .. 84.93326568603516,False,[2706317525385216],2706317525385216,True,V_CMB,27063

In [108]:
merged.rename_column("TARGETID","OBJ_ID")

In [71]:
## let us cross match to get PSFSIZE!

In [109]:
merged.write("/pscratch/sd/v/virajvm/legacy_galaxies/catalogs/galaxies.fits", overwrite=True)

In [110]:
merged

OBJ_ID,SURVEY,PROGRAM,HEALPIX,Z,DELTACHI2,ZWARN,Z_CMB,RA,DEC,RA_TARGET,DEC_TARGET,DESINAME,LUMI_DIST_MPC,LOG_MSTAR_M24,LOG_MSTAR_M24_ERR,MAG_G,MAG_R,MAG_Z,MAG_G_TARGET,MAG_R_TARGET,MAG_Z_TARGET,SAMPLE,DWARF_MASKBIT,MSTAR_MASKBIT,MAG_TYPE,PHOTOMETRY_UPDATED,R50_R,SHAPE_PARAMS,IN_SGA_2020,ASSOCIATED_TARGETIDS,DWARF_PRIMARY_TARGETID,DWARF_PRIMARY,DIST_SOURCE,PROPERTY_SOURCE_TARGETID,BRICKNAME,PSFSIZE_G,PSFSIZE_R,PSFSIZE_Z
int64,bytes7,bytes6,int32,float64,float64,int32,float64,float64,float64,float64,float64,bytes22,float64,float64,float64,float64,float64,float64,float64,float64,float64,bytes10,int32,int32,bytes13,bool,float64,float64[2],bool,object,int64,bool,bytes10,int64,bytes8,float32,float32,float32
2706072053743616,main,dark,27162,0.02067871920862875,51.423833825741895,0,0.02185958059324311,147.4777724465266,-4.676317817295627,147.4777724465266,-4.676317817295627,DESI J147.4777-04.6763,98.4670639038086,8.338191032409668,0.02161700759700041,19.767858505249023,19.131961822509766,18.707191467285156,19.767858505249023,19.131961822509766,18.707191467285156,LOWZ,0,0,TRACTOR_OG,False,3.3400158882141113,0.8321884870529175 .. -77.67112731933594,False,[2706072053743616],2706072053743616,True,V_CMB,2706072053743616,1475m047,1.8973777,1.4541191,1.1051942
2706102982541312,main,dark,26141,0.017032653316364645,52.587958217802225,0,0.018156770780494602,194.32707696171147,-3.5259274788974393,194.32707696171147,-3.5259274788974393,DESI J194.3270-03.5259,81.5626220703125,8.096750259399414,0.02525354129101601,19.9845027923584,19.363792419433594,18.983266830444336,19.9845027923584,19.363792419433594,18.983266830444336,LOWZ,0,0,TRACTOR_OG,False,3.5016419887542725,0.8836742639541626 .. -33.543739318847656,False,[2706102982541312],2706102982541312,True,V_CMB,2706102982541312,1943m035,1.3323532,1.3068316,1.2971201
2706154996105216,main,dark,23102,0.026127110560297866,78.27858669866691,0,0.02564987549513864,55.44840405171893,-1.1841882389875291,55.44840405171893,-1.1841882389875291,DESI J055.4484-01.1841,115.86506652832031,8.417013168334961,0.015496369564586361,20.011428833007812,19.33967399597168,18.971805572509766,20.011428833007812,19.33967399597168,18.971805572509766,LOWZ,0,0,TRACTOR_OG,False,3.5959646701812744,0.47487756609916687 .. 63.821685791015625,False,[2706154996105216],2706154996105216,True,V_CMB,2706154996105216,0553m012,1.7105765,1.2939546,1.2036133
2706186377887744,main,dark,21865,0.03483788702310787,88.01792028345335,0,0.035780520361125,125.93542728709848,-0.0590019387959595,125.93542728709848,-0.0590019387959595,DESI J125.9354000.-590,162.83070373535156,8.681873321533203,0.018363662534729992,19.930143356323242,19.27661895751953,18.882261276245117,19.930143356323242,19.27661895751953,18.882261276245117,LOWZ,0,0,TRACTOR_OG,False,3.67437744140625,0.40696120262145996 .. -47.35565948486328,False,[2706186377887744],2706186377887744,True,V_CMB,2706186377887744,1258p000,1.3402432,1.2273034,1.3054327
2706247711195136,main,dark,27653,0.020029654061365473,40.72983623316395,0,0.0212375641784448,181.53274336777432,2.4489531897059846,181.53274336777432,2.4489531897059846,DESI J181.5327+02.4489,95.62100982666016,8.326117515563965,0.023986635610781366,19.847965240478516,19.18031120300293,18.807758331298828,19.847965240478516,19.18031120300293,18.807758331298828,LOWZ,0,0,TRACTOR_OG,False,3.303351402282715,0.8462779521942139 .. -67.152587890625,False,[2706247711195136],2706247711195136,True,V_CMB,2706247711195136,1816p025,2.330694,1.3881552,1.2428343
2706317525385216,main,dark,17899,0.016681511342151477,99.7627397031174,0,0.015739788281099898,29.443161274980465,5.4525645136574665,29.443161274980465,5.4525645136574665,DESI J029.4431+05.4525,70.57759094238281,8.093667984008789,0.017058261913048385,19.649059295654297,19.029748916625977,18.647863388061523,19.649059295654297,19.029748916625977,18.647863388061523,LOWZ,0,0,TRACTOR_OG,False,3.2249057292938232,0.7416470646858215 .. 84.93326568603516,False,[2706317525385216],2706317525385216,True,V_CMB,2706317